In [8]:
from google.colab import drive
drive.mount('/content/drive')

!pip install implicit -q

import numpy as np
import pandas as pd
import scipy.sparse as sp
import pickle
import os
import joblib

DRIVE_BASE = '/content/drive/MyDrive/anime_recommender/processed'
MODEL_DIR = '/content/drive/MyDrive/anime_recommender/model'
os.makedirs(MODEL_DIR, exist_ok=True)

print(os.listdir(DRIVE_BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['train_ratings.parquet', 'test_ratings.parquet', 'anime_meta.parquet', 'content_matrix.npy', 'train_matrix.npz', 'anime_similarity.npy', 'mappings.pkl', 'eda.png']


In [9]:
with open(f'{DRIVE_BASE}/mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)

user2idx = mappings['user2idx']
anime2idx = mappings['anime2idx']
idx2user = mappings['idx2user']
idx2anime = mappings['idx2anime']
unique_users = mappings['unique_users']
unique_anime = mappings['unique_anime']

train_matrix = sp.load_npz(f'{DRIVE_BASE}/train_matrix.npz')
train_df = pd.read_parquet(f'{DRIVE_BASE}/train_ratings.parquet')
test_df = pd.read_parquet(f'{DRIVE_BASE}/test_ratings.parquet')
anime_meta = pd.read_parquet(f'{DRIVE_BASE}/anime_meta.parquet')
anime_similarity = np.load(f'{DRIVE_BASE}/anime_similarity.npy')
content_matrix = np.load(f'{DRIVE_BASE}/content_matrix.npy')

print("train_matrix shape:", train_matrix.shape)
print("train ratings:", len(train_df))
print("test ratings:", len(test_df))
print("anime_meta shape:", anime_meta.shape)
print("anime_similarity shape:", anime_similarity.shape)

train_matrix shape: (223565, 10502)
train ratings: 44517772
test ratings: 11129443
anime_meta shape: (10502, 17)
anime_similarity shape: (10502, 10502)


In [10]:
# Compute user mean ratings from training data
user_mean_ratings = train_df.groupby('user_idx')['rating'].mean().to_dict()

n_users, n_anime = train_matrix.shape
user_means = np.array([user_mean_ratings.get(i, 7.0) for i in range(n_users)], dtype=np.float32)

print(f"User mean rating - avg: {user_means.mean():.3f}, std: {user_means.std():.3f}")

# Mean-center the sparse matrix while keeping it sparse
# Only modifies stored (non-zero = rated) values
train_matrix_centered = train_matrix.astype(np.float32).tocsr()

for i in range(n_users):
    start = train_matrix_centered.indptr[i]
    end = train_matrix_centered.indptr[i+1]
    if end > start:
        train_matrix_centered.data[start:end] -= user_means[i]

print(f"Centered - mean of stored values: {train_matrix_centered.data.mean():.4f}")
print(f"Centered - std: {train_matrix_centered.data.std():.4f}")

# Shift to make all stored values non-negative (ALS needs positive values)
shift = float(abs(train_matrix_centered.data.min()) + 0.01)
train_matrix_final = train_matrix_centered.copy()
train_matrix_final.data += shift

print(f"Shift value: {shift:.6f}")
print(f"After shift - min: {train_matrix_final.data.min():.4f}, max: {train_matrix_final.data.max():.4f}")

User mean rating - avg: 7.713, std: 0.830
Centered - mean of stored values: 0.0000
Centered - std: 1.4306
Shift value: 8.992349
After shift - min: 0.0100, max: 17.9536


In [21]:
import implicit

# implicit expects item x user matrix
train_for_als = train_matrix_final.T.tocsr().astype(np.float32)

print("Matrix fed to ALS (item x user):", train_for_als.shape)

model = implicit.cpu.als.AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=30,
    random_state=42
)

model.fit(train_for_als)

user_factors = model.item_factors   # shape: (n_users, k)
item_factors = model.user_factors   # shape: (n_anime, k)

print("user_factors shape:", user_factors.shape)
print("item_factors shape:", item_factors.shape)

Matrix fed to ALS (item x user): (10502, 223565)


  0%|          | 0/30 [00:00<?, ?it/s]

user_factors shape: (223565, 50)
item_factors shape: (10502, 50)


In [22]:
# Recompute anime_similarity to match filtered model size
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

n_anime = item_factors.shape[0]
print(f"Model has {n_anime} anime, recomputing similarity matrix for this size...")

content_matrix_filtered = content_matrix[:n_anime].astype(np.float32)
anime_similarity = cos_sim(content_matrix_filtered)

print("anime_similarity shape:", anime_similarity.shape)
# Should now print (10502, 10502)

Model has 10502 anime, recomputing similarity matrix for this size...
anime_similarity shape: (10502, 10502)


In [23]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Filter test set to only indices that exist in the trained model
valid_user_indices = set(range(user_factors.shape[0]))
valid_anime_indices = set(range(item_factors.shape[0]))

test_df_valid = test_df[
    test_df['user_idx'].isin(valid_user_indices) &
    test_df['anime_idx'].isin(valid_anime_indices)
].copy()

print(f"Test set before filter: {len(test_df)}")
print(f"Test set after filter:  {len(test_df_valid)}")

# Sample from valid test set
test_sample = test_df_valid.sample(min(50000, len(test_df_valid)), random_state=42)

predictions = []
actuals = []

for _, row in test_sample.iterrows():
    u_idx = int(row['user_idx'])
    a_idx = int(row['anime_idx'])
    actual = float(row['rating'])

    # Raw ALS score
    raw_score = np.dot(user_factors[u_idx], item_factors[a_idx])

    # Reverse shift, add user mean back to get back to 1-10 scale
    pred = (raw_score - shift) + user_means[u_idx]
    pred = np.clip(pred, 1.0, 10.0)

    predictions.append(pred)
    actuals.append(actual)

rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae = mean_absolute_error(actuals, predictions)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

# --- Precision@K and Recall@K ---
print("\nComputing Precision@10 and Recall@10...")

RELEVANCE_THRESHOLD = 7
K = 10

test_grouped = test_df_valid.sample(min(50000, len(test_df_valid)), random_state=42).groupby('user_idx')
precisions = []
recalls = []

for user_idx, user_test in test_grouped:
    if len(user_test) < 5:
        continue

    relevant_items = set(user_test[user_test['rating'] >= RELEVANCE_THRESHOLD]['anime_idx'].tolist())
    if len(relevant_items) == 0:
        continue

    test_anime_idxs = user_test['anime_idx'].tolist()
    scores = []
    for a_idx in test_anime_idxs:
        raw = np.dot(user_factors[user_idx], item_factors[a_idx])
        pred = np.clip((raw - shift) + user_means[user_idx], 1.0, 10.0)
        scores.append((a_idx, pred))

    top_k_items = set([x[0] for x in sorted(scores, key=lambda x: x[1], reverse=True)[:K]])
    hits = len(top_k_items & relevant_items)

    precisions.append(hits / K)
    recalls.append(hits / len(relevant_items))

p_at_k = np.mean(precisions)
r_at_k = np.mean(recalls)
print(f"Precision@{K}: {p_at_k:.4f}")
print(f"Recall@{K}:    {r_at_k:.4f}")

Test set before filter: 11129443
Test set after filter:  11129443
RMSE: 6.6883
MAE:  6.4738

Computing Precision@10 and Recall@10...
Precision@10: 0.3596
Recall@10:    0.9986


In [24]:
def get_recommendations(
    user_idx,
    train_matrix_original,
    user_factors,
    item_factors,
    user_means,
    shift,
    anime_similarity,
    anime_meta,
    idx2anime,
    top_n=10,
    exclude_watched=True
):
    n_anime = item_factors.shape[0]

    # --- CF scores ---
    cf_raw = np.dot(user_factors[user_idx], item_factors.T)
    cf_scores = np.clip((cf_raw - shift) + user_means[user_idx], 1.0, 10.0)
    cf_norm = (cf_scores - cf_scores.min()) / (cf_scores.max() - cf_scores.min() + 1e-8)

    # --- Content-based scores ---
    user_row = train_matrix_original[user_idx]
    rated_indices = user_row.indices
    # Filter rated_indices to only those within n_anime range
    rated_indices = rated_indices[rated_indices < n_anime]
    rated_values = np.array(user_row.data, dtype=np.float32)[:len(rated_indices)]

    if len(rated_indices) > 0:
        weights = rated_values - user_means[user_idx]
        weights = np.clip(weights, 0, None)

        if weights.sum() > 0:
            weights = weights / weights.sum()
            cb_scores = np.zeros(n_anime, dtype=np.float32)
            for sim_idx, w in zip(rated_indices, weights):
                cb_scores += w * anime_similarity[sim_idx]
        else:
            cb_scores = anime_similarity[rated_indices].mean(axis=0)
    else:
        cb_scores = np.zeros(n_anime, dtype=np.float32)

    cb_norm = (cb_scores - cb_scores.min()) / (cb_scores.max() - cb_scores.min() + 1e-8)

    # --- Adaptive alpha ---
    n_rated = len(rated_indices)
    if n_rated < 20:
        alpha = 0.3
    elif n_rated < 100:
        alpha = 0.5
    else:
        alpha = 0.7

    # --- Hybrid score ---
    hybrid = alpha * cf_norm + (1 - alpha) * cb_norm

    # --- Exclude watched ---
    if exclude_watched and len(rated_indices) > 0:
        hybrid[rated_indices] = -1.0

    # --- Top N ---
    top_indices = np.argsort(hybrid)[::-1][:top_n + 20]

    results = []
    for a_idx in top_indices:
        if hybrid[a_idx] < 0:
            continue
        original_id = idx2anime[a_idx]
        row = anime_meta[anime_meta['anime_id'] == original_id]
        if row.empty:
            continue
        info = row.iloc[0]
        results.append({
            'anime_id': original_id,
            'name': info['display_name'],
            'genres': info['Genres'],
            'type': info['Type'],
            'community_score': info['Score'],
            'hybrid_score': round(float(hybrid[a_idx]), 4),
            'cf_score': round(float(cf_scores[a_idx]), 4),
            'cb_score': round(float(cb_norm[a_idx]), 4),
            'alpha_used': alpha
        })
        if len(results) >= top_n:
            break

    return pd.DataFrame(results)

# Test
recs = get_recommendations(
    5, train_matrix, user_factors, item_factors,
    user_means, shift, anime_similarity, anime_meta, idx2anime, top_n=10
)
print(recs[['name', 'genres', 'hybrid_score', 'cf_score']].to_string())

                                            name                                                       genres  hybrid_score  cf_score
0          My Teen Romantic Comedy SNAFU Climax!                Slice of Life, Comedy, Drama, Romance, School        0.4940       1.0
1             My Teen Romantic Comedy SNAFU TOO!                Slice of Life, Comedy, Drama, Romance, School        0.4926       1.0
2                      The Pet Girl of Sakurasou                Slice of Life, Comedy, Drama, Romance, School        0.4921       1.0
3                      Little Busters! ~Refrain~  Slice of Life, Comedy, Supernatural, Drama, Romance, School        0.4905       1.0
4              Love, Chunibyo & Other Delusions!                Slice of Life, Comedy, Drama, Romance, School        0.4888       1.0
5                Canvas 2:Rainbow Colored Sketch                        Comedy, Drama, Romance, Slice of Life        0.4885       1.0
6                                 Kokoro Connect  Slice of Lif

In [25]:
joblib.dump(model, f'{MODEL_DIR}/als_model.pkl')
np.save(f'{MODEL_DIR}/user_factors.npy', user_factors)
np.save(f'{MODEL_DIR}/item_factors.npy', item_factors)
np.save(f'{MODEL_DIR}/user_means.npy', user_means)
np.save(f'{MODEL_DIR}/anime_similarity_filtered.npy', anime_similarity)  # save the recomputed one

model_config = {
    'factors': 50,
    'shift': shift,
    'rmse': rmse,
    'mae': mae,
    'precision_at_10': p_at_k,
    'recall_at_10': r_at_k,
    'relevance_threshold': RELEVANCE_THRESHOLD,
    'n_users': n_users,
    'n_anime': n_anime,
}

with open(f'{MODEL_DIR}/model_config.pkl', 'wb') as f:
    pickle.dump(model_config, f)

print("Saved:", os.listdir(MODEL_DIR))

Saved: ['als_model.pkl', 'user_factors.npy', 'item_factors.npy', 'user_means.npy', 'anime_similarity_filtered.npy', 'model_config.pkl']


In [26]:
from google.colab import files

files_to_download = [
    f'{MODEL_DIR}/als_model.pkl',
    f'{MODEL_DIR}/user_factors.npy',
    f'{MODEL_DIR}/item_factors.npy',
    f'{MODEL_DIR}/user_means.npy',
    f'{MODEL_DIR}/user_means.npy',
    f'{MODEL_DIR}/anime_similarity_filtered.npy',
    f'{MODEL_DIR}/model_config.pkl',
]

for filepath in files_to_download:
    print(f"Downloading {os.path.basename(filepath)}...")
    files.download(filepath)

print("\nAlso download from processed folder:")
print("- anime_meta.parquet")
print("- mappings.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Also download from processed folder:
- anime_meta.parquet
- mappings.pkl
